# 00 - Setup del proyecto
Prepara el entorno del proyecto de predicción de demanda horaria del STM:
1. Hace importable el paquete `bondis` (carpeta `src/`).
2. Crea el esquema `workspace.bondis` y el volume `raw` en Unity Catalog.
3. Crea las subcarpetas del volume para cada fuente de datos.
4. Verifica el entorno y muestra la división temporal acordada.

Es **idempotente**: puede ejecutarse varias veces sin efectos secundarios.

Los demás notebooks lo invocan al inicio con `%run ./00_setup`.


## 1. Hacer importable el paquete `bondis`

In [0]:
import importlib
import os
import sys
 
def _buscar_raiz_repo(inicio):
    """Sube carpeta por carpeta desde 'inicio' hasta encontrar src/bondis.
 
    No asume dónde está el notebook: funciona tanto si se ejecuta directo
    como si se invoca con %run desde otro notebook u otra carpeta.
    """
    actual = os.path.abspath(inicio)
    while True:
        if os.path.isdir(os.path.join(actual, "src", "bondis")):
            return actual
        padre = os.path.dirname(actual)
        if padre == actual:
            return None
        actual = padre
 
 
_contexto = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_notebook_path = "/Workspace" + _contexto.notebookPath().get()
 
REPO_ROOT = _buscar_raiz_repo(os.getcwd()) or _buscar_raiz_repo(os.path.dirname(_notebook_path))
if REPO_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta src/bondis.\n"
        f"  Directorio de trabajo: {os.getcwd()}\n"
        f"  Notebook: {_notebook_path}\n"
        "Verificar que el notebook esté dentro del Git folder del repo."
    )
SRC_PATH = os.path.join(REPO_ROOT, "src")
 
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
 
from bondis import config
 
importlib.reload(config)  # toma cambios recientes de config.py sin reiniciar
 
print("Raíz del repo:      ", REPO_ROOT)
print("Ruta de código:     ", SRC_PATH)

## 2. Crear esquema y volume en Unity Catalog

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}.{config.VOLUME}"
)
 
print(f"Esquema listo: {config.CATALOG}.{config.SCHEMA}")
print(f"Volume listo:  {config.RAW_PATH}")

## 3. Crear subcarpetas del volume

Cada fuente de datos va en su propia carpeta:
| Carpeta | Contenido |
|---|---|
| `viajes_zip/` | ZIP mensuales de viajes subidos desde local |
| `viajes/` | CSV mensuales de viajes del STM |
| `clima/` | Lluvia y temperatura por hora |
| `calendario/` | Feriados y calendarios de ANEP y UdelaR |
| `paros/` | Días y horas afectados por paros |

In [0]:
for carpeta in [config.RAW_VIAJES_ZIP, config.RAW_VIAJES, config.RAW_CLIMA, config.RAW_CALENDARIO, config.RAW_PAROS]:
    dbutils.fs.mkdirs(carpeta)
    print("Carpeta lista:", carpeta)

## 4. Verificación del entorno

In [0]:
print("Versión de Spark:", spark.version)
print("Usuario actual:  ", spark.sql("SELECT current_user()").first()[0])
print()
 
for carpeta in [config.RAW_VIAJES_ZIP, config.RAW_VIAJES, config.RAW_CLIMA, config.RAW_CALENDARIO, config.RAW_PAROS]:
    archivos = [f for f in dbutils.fs.ls(carpeta) if not f.isDir()]
    total_mb = sum(f.size for f in archivos) / 1e6
    print(f"{carpeta}: {len(archivos)} archivo(s), {total_mb:.1f} MB")
    for f in archivos:
        print(f"    - {f.name}  ({f.size / 1e6:.1f} MB)")

In [0]:
display(spark.sql(f"SHOW TABLES IN {config.CATALOG}.{config.SCHEMA}"))

## 5. División temporal acordada

Intervalos semiabiertos `[inicio, fin)`. Filtrar siempre con `fecha >= inicio` y `fecha < fin`.

In [0]:
print(f"{'Conjunto':<8} {'Inicio':<12} {'Fin (excl.)':<12}")
for nombre, (inicio, fin) in config.PARTICIONES.items():
    print(f"{nombre:<8} {inicio:<12} {fin:<12}")
 
print()
print(f"Horizonte de predicción: {config.HORIZONTE_DIAS} día(s)")
print(f"Semilla: {config.SEED}")